In [ ]:
import torch
import string 
import unicodedata

In [ ]:

device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda") 
torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")


In [ ]:
allowed_chars = string.ascii_letters + " .,;'" + "_"
n_lets = len(allowed_chars) 

def unicodeToAscii(s): 
    return ''.join(
        c for c in unicodedata.normalize('NFD', s) 
        if unicodedata.category(c) != 'Mn'
        and c if allowed_chars
    )

###
print(f"Converting 'Slusarski' to {unicodeToAscii('Slusarski')}")
###

In [ ]:
def letter_to_indx(letter): 
    if letter not in allowed_chars:
        return allowed_chars.find("_") 
    else: 
        return allowed_chars.find(letter) 

def lineToTensor(line): 
    tensor = torch.zeros(len(line), 1, n_lets)
    for li, letter in enumerate(line): 
        tensor[li][0][letter_to_indx(letter)] = 1
    return tensor 


print(f"The letter 'a' becomes{lineToTensor('a')}")
print(f"The name 'Ahn' becomes {lineToTensor('Ahn')}")


In [ ]:
from io import open 
import glob 
import os 
import time 
import torch 

from torch.utils.data import Dataset 

class NamesDataSet(Dataset):

    def __init__(self, data_dir): 
        self.data_dir = data_dir
        self.load_time = time.localtime
        labels_set = set() 

        self.data = [] 
        self.data_tensors = [] 
        self.labels = [] 
        self.labels_tensors = [] 

        text_files = glob.glob(os.path.join(data_dir, '*.txt')) 

        for filename in text_files: 
            label = os.path.splittext(os.path.basename(filename))[0]
            labels_set.add(label) 
            lines = open(filename, encoding='utf-8').read().strip().split('\n')
            for name in lines: 
                self.data.append(name)
                self.data_tensors.append(lineToTensor(name))
                self.labels.append(label) 

        self.labels_uniq = sorted(list(labels_set)) 
        for label in self.labels: 
            label_idx = self.labels_uniq.index(label) 
            tmp_tensor = torch.tensor([label_idx], dtype=torch.long) 
            self.labels_tensors.append(tmp_tensor) 


    def __len__(self): 
        return len(self.data) 

    def __getitem__(self, idx):
        data_item = self.data[idx] 
        data_label = self.labels[idx]
        data_tensor = self.data_tensors[idx] 
        label_tensor = self.label_tensor[idx] 

        return label_tensor, data_tensor, data_label, data_item

    


In [ ]:
alldata = NamesDataSet("data/names")
print(f"loaded {len(alldata)} items of data")
print(f"example = {alldata[0]}")


In [ ]:
train_set, test_set = torch.utils.data.random_split(alldata, [.85, .15], generator=torch.Generator(device=device).manual_seed(2024))
print(f"train exampels = {len(train_set)}, validation examples = {len(test_set)}")


In [ ]:
import torch.nn as nn 
import torch.nn as functional 

class CharRNN(nn.Module):
    def __init__(self, inp_size, hid_size, outp_size): 
        super(CharRNN, self).__init__() 

        self.rnn = nn.RNN(inp_size, hid_size)
        self.h2o = nn.Linear(hid_size, outp_size) 
        self.softmax = nn.LogSoftmax(dim=1) 

    def forward(self, line_tensr): 
        rnn_out, hidden  = self.rnn(line_tensr)
        outp = self.h2o(hidden[0]) 
        outp = self.softmax(outp)

        return outp

n_hidden = 128 
rnn = CharRNN(n_letters, n_hidden, len(alldata.labels_uniq))
print(rnn) 


